# Data View

In [ ]:
import pandas as pd
import sys
sys.path.append("/home/ec2-user/studies/HUIFA-RNA-SEQ-new/phenoAI/")

from phenokit.preprocessing.qc import MissingValueAnalyzer
from phenokit.preprocessing.io import read_parquet

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/sleep/sleep.parquet'
df_sleep = read_parquet(pfile)
print(df_sleep.shape)
df_sleep.head()

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/sleep/hrv.parquet'
df_hrv = read_parquet(pfile)
print(df_hrv.shape)
df_hrv.head()

In [ ]:
cfile = '/home/ec2-user/studies/hpp_datasets/sleep/metadata/sleep_data_dictionary.csv'
df_sleep_dict = pd.read_csv(cfile)
df_sleep_dict.head()

In [ ]:
missing_analyzer = MissingValueAnalyzer(df_sleep)
missing_analyzer.overall_missing_rate()
missing_analyzer.report_features()
missing_analyzer.report_samples()
missing_analyzer.report_heatmap()

# Data Filtering

In [ ]:
df_pcount = df_sleep['participant_id'].value_counts().to_frame()
pvalid = df_pcount[df_pcount['count'] >= 3].index.to_list()
df_sleep_selected = df_sleep[df_sleep['participant_id'].isin(pvalid)]

In [ ]:
sleep_metadata_col = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_timestamp', 'collection_date', 'night', 'warnings',
                        'timezone', 'device_sn', 'hardware_version', 'software_version', 'device_model', 'channels_edf_file', 'channels_time_series', 
                        'events_time_series', 'study_start_time', 'study_end_time']
sleep_metadata_col = [col for col in sleep_metadata_col if col in df_sleep.columns]
feature_col = [col for col in df_sleep.columns if col not in sleep_metadata_col]

In [ ]:
# 1) 确保时间列是 datetime（很重要）
df_sleep_selected = df_sleep_selected.copy()
df_sleep_selected["collection_timestamp"] = pd.to_datetime(
    df_sleep_selected["collection_timestamp"], errors="coerce"
)

# 2) 选出每组的“第一个时间点”的元数据行
meta_keep = ["participant_id", "cohort", "research_stage", "collection_timestamp", "collection_date"]
meta_keep = [c for c in meta_keep if c in df_sleep_selected.columns]

df_meta_first = (
    df_sleep_selected
    .sort_values("collection_timestamp")
    .groupby(["participant_id", "research_stage"], as_index=False)
    .first()[meta_keep]
)

# 3) 计算每组数值特征的中位数（忽略 NaN）
df_feat_median = (
    df_sleep_selected
    .groupby(["participant_id", "research_stage"], as_index=False)[feature_col]
    .median()
)

# 4) 合并：元数据（最早时间点） + 特征（中位数）
df_sleep_agg = df_meta_first.merge(
    df_feat_median,
    on=["participant_id", "research_stage"],
    how="inner",
    validate="1:1"
)

In [ ]:
metadata_col = ['participant_id', 'cohort', 'research_stage', 'collection_date', 'collection_timestamp']
df_sleep_metadata = df_sleep_agg[metadata_col]
df_sleep_features = df_sleep_agg.drop(columns=metadata_col)

In [ ]:
import numpy as np

missing_rate = df_sleep_features.isna().mean(axis=0)
df_sleep_feature_filtering = df_sleep_features.loc[:, missing_rate < 0.5]
df_sleep_feature_filtering = df_sleep_feature_filtering.astype(np.float32)

df_sleep_feature_imputed = df_sleep_feature_filtering.fillna(df_sleep_feature_filtering.mean())

# Save

In [ ]:
df_sleep_metadata.reset_index(drop=True, inplace=True)
df_sleep_feature_imputed.reset_index(drop=True, inplace=True)
df_sleep = pd.concat([df_sleep_metadata, df_sleep_feature_imputed], axis=1).dropna(axis=1)
pfile = 'pheno_data/202512/007-sleep_imputed.parquet'
df_sleep.to_parquet(pfile, index=False)

In [ ]:
pfile = 'pheno_data/202512/007-sleep_imputed.parquet'
df_sleep = pd.read_parquet(pfile)
print(df_sleep.shape)
df_sleep.head()